# Explore spectral analysis OPM LID project

## 0) Import

In [ ]:
# general packages
import json
import os
import importlib
import sys
import numpy as np
import pandas as pd
from itertools import compress
import matplotlib.pyplot as plt

# ephys packages
import mne
from mne.time_frequency import psd_array_welch

In [ ]:
def add_repo_dir():
    """adds local repo directory to sys to allow importing from repo"""

    wd = os.getcwd()

    COUNTER = 20  #  to prevent eternal while loop

    while not wd.endswith('lid_opm'):
        wd = os.path.dirname(wd)
        COUNTER -= 1

        if COUNTER == 0:
            raise ValueError('repo dir not found!')

    print(f'add repo directory to sys: {wd} ')

    sys.path.append(wd)

In [ ]:
# add custom functions

add_repo_dir()

import utils.load_utils as load_utils
# from source_raw_conversion import load_PTB_source_opm as source_opm
import plotting.processing_checks as proc_plotting
import utils.load_processed_data as load_proc_data

## 1. Define settings, load data


Define recording to analyze

In [ ]:
CONFIG_VERSION = "v1"

SUB = '95'  # 
SES = 'Dec'
TASK = 'gonogo3'
ACQ = 'testArd'

Import settings

In [ ]:
importlib.reload(load_utils)

# load settings
sub_config = load_utils.load_subject_config(subject_id=SUB,)
preproc_config = load_utils.load_preproc_config(version=CONFIG_VERSION,)

try:
    sub_meta_info = load_utils.get_sub_rec_metainfo(config_sub=sub_config)
except FileNotFoundError:
    print('WARNING: no rec admin availbale')
    sub_meta_info = None




In [ ]:
importlib.reload(load_proc_data)

meg_raw, aux_raw = load_proc_data.load_cleaned_data(
    SUB=SUB,
    SES=SES,
    ACQ=ACQ,
    TASK=TASK,
    load_aux=True,
    data_type='epochs',
    config_version=CONFIG_VERSION,
)

In [ ]:
aux_raw

### select neural data for behavioral events

- select only Z for spectral analysis
- split hemispheres before combining with behavior
- consider only motor channels (arbitrary based on topogram)



In [ ]:
# Split hemispheres 

# meg_clean_left = raw_clean.copy().pick_channels([ch for ch in raw_clean.ch_names if ch[0] == 'L'])
# meg_clean_right = raw_clean.copy().pick_channels([ch for ch in raw_clean.ch_names if ch[0] == 'R'])


In [ ]:
MOTOR_CH_NRS = ['206', '207', '305', '306', '404', '405']
event_code_dict = {'go': 1, 'nogo': 2, 'abort': 3}


In [ ]:
ONLY_MOTOR = False

meg_channels = {'y': [ch_name for ch_name in raw_clean.ch_names
                      if '_by' in ch_name],
                'z': [ch_name for ch_name in raw_clean.ch_names
                      if '_bz' in ch_name]}

all_meg_channels = meg_channels['y'] + meg_channels['z']


# Select the MEG data from the raw file
rawz = raw_clean.copy().pick(meg_channels['z'])
cleanz = move_cleaned1.copy().pick(meg_channels['z'])

# # find channels present in both lists
# meg_z_motor_channels = [ch for ch in motor_channels if ch in meg_channels['z']]

if ONLY_MOTOR:
    motor_channels = [ch_name for ch_name in rawz.ch_names for m in MOTOR_CH_NRS
                        if any([m in ch_name])]

    rawz = rawz.copy().pick(motor_channels)
    cleanz = cleanz.copy().pick(motor_channels)



In [ ]:

# quick check of epochs, only Z, only motor channels
# zmotor_epochs = meg_epochs.copy().pick(meg_z_motor_channels)

# meg_epochs.plot()
epochs_clean['go_left'].average().plot()
epochs_clean['go_left'].plot_psd()

## visualize over events

check epochs: are muscular artefacts (bad segments/annotations) excluded or still included?
- if excluded: how are NaNs handled, are events with bad-segments/missings excluded?

In [ ]:
T_POST = 3


event_samples = [np.where(t == move_cleaned1.times)[0][0] for t in FL_trigger_times]
event_codes = [event_code_dict[k] for k in FL_trigger_types]

event_arr = np.concatenate(
    [[event_samples, np.zeros(len(event_samples), dtype=int), event_codes]]
).T


epochs_z = mne.Epochs(
    rawz.copy(), events=event_arr, event_id=event_code_dict,
    tmin=-.5, tmax=T_POST, baseline=None,
)
epochs_zclean = mne.Epochs(
    cleanz.copy(), events=event_arr, event_id=event_code_dict,
    tmin=-.5, tmax=T_POST, baseline=None,
)

In [ ]:
# TFR
def get_mean_tf_epochs(temp_epochs):

    freqs = np.arange(2, 95, 1)  # Generate frequencies from 6 to 45 Hz in 1 Hz steps
    n_cycles = freqs / 2.0  # different number of cycle per frequency

    tfr = temp_epochs['go'].compute_tfr(
        method="multitaper",
        freqs=freqs,
        n_cycles=n_cycles,
        average=True,
        return_itc=False,
        # decim=3,
    )
    mean_tf = np.mean(tfr.get_data(), axis=0)

    for i in np.arange(mean_tf.shape[0]):
        mean_tf[i, :] = (mean_tf[i, :] - np.mean(mean_tf[i, :])) / np.std(mean_tf[i, :])

    return mean_tf

In [ ]:
%matplotlib qt

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))



tf_times = np.arange(-.5, T_POST + (1/epochs_zclean.info['sfreq']), 1/epochs_zclean.info['sfreq'])

im = ax.pcolormesh(
    tf_times,
    freqs,
    mean_tf.T,
    shading="auto",
    cmap="coolwarm",
    vmin=-2, vmax=2,
)

ax.axvline(0, color="w", linestyle="--", linewidth=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("Time–Frequency Power")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Power")

plt.tight_layout()

plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 5))

def plot_tf_ax(ax, temp_epochs, temp_mean_tf=None):

    if type(temp_mean_tf) == None: temp_mean_tf = get_mean_tf_epochs(temp_epochs)

    tf_times = np.arange(-.5, T_POST + (1/temp_epochs.info['sfreq']), 1/temp_epochs.info['sfreq'])

    im = ax.pcolormesh(
        tf_times,
        freqs,
        temp_mean_tf,
        shading="auto",
        cmap="coolwarm",
        vmin=-2, vmax=2,
    )

    ax.axvline(0, color="w", linestyle="--", linewidth=1)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title("Time–Frequency Power")

    # cbar = ax.colorbar(im, ax=ax)
    # cbar.set_label("Power")

    return ax

# plt.tight_layout()

# plt.show()

In [ ]:

event_code_dict = {'go': 1, 'nogo': 2, 'abort': 3}

event_samples = [np.argmin(abs(t - acc.times))
                    for t in AN_trig_times]
event_codes = [event_code_dict[k] for k in FL_trigger_types]

event_arr = np.concatenate(
    [[event_samples, np.zeros(len(event_samples), dtype=int), event_codes]]
).T
epochs_acc = mne.Epochs(
    acc, events=event_arr, event_id=event_code_dict,
    tmin=-.5, tmax=T_POST, baseline=None,
)

In [ ]:
%matplotlib inline

In [ ]:
mean_tf = get_mean_tf_epochs(epochs_z)
mean_tf_clean = get_mean_tf_epochs(epochs_zclean)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharey='row')

epochs_z['go'].average().plot(axes=axes[0, 0], show=False,)
epochs_zclean['go'].average().plot(axes=axes[0, 1], show=False,)

plot_tf_ax(ax=axes[1, 0], temp_epochs=epochs_z['go'].copy(), temp_mean_tf=mean_tf)
plot_tf_ax(ax=axes[1, 1], temp_epochs=epochs_zclean['go'].copy(), temp_mean_tf=mean_tf_clean)


epochs_acc['go'].average(picks='misc').plot(axes=axes[2, 0], show=False,)
epochs_acc['go'].average(picks='misc').plot(axes=axes[2, 1], show=False,)


plt.tight_layout()

plt.show()

